In [56]:
# load data science, stat and plot libs & time analysis anomaly detection
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns
from datetime import datetime
import yaml


## load data

In [57]:
# load config: periods and trigger dates
with open('analysis_config.yaml', 'r') as f:
    config = yaml.safe_load(f)
periods = {name: (pd.to_datetime(info['start']), pd.to_datetime(info['end'])) for name, info in config['periods'].items()}
trigger_dates_dt = [pd.to_datetime(d) for d in config['trigger_dates']]
#
# call logs processed -- see processing.ipynb for details
df = pd.read_csv('data-for-analysis.csv')


Note: maybe non-EU unknown call we can ignore, since it is mostly Israel and there it is common to spam.. ALL hypotetically "interesting call were from EU".

## 1. Baseline Statistics: μ (mean intensity)

In [58]:
# Convert date column to datetime
df['date_dt'] = pd.to_datetime(df['date'], errors='coerce')

# Data already contains:
# - analysis_period: categorization into Baseline/Phase 1/2/3
# - trigger_48h, trigger_7d, trigger_1M: boolean flags for trigger windows
# - is_known: True for known contacts, False for unknown
# - is_EU: True for EU, False for non-EU, None for unknown
# - country: country code or 'Non-EU'
# - duration_sec: call duration in seconds

print(f"✓ Data loaded successfully")
print(f"Total calls: {len(df)}")
print(f"Date range: {df['date_dt'].min()} to {df['date_dt'].max()}")
print(f"\nColumns: {df.columns.tolist()}")

✓ Data loaded successfully
Total calls: 1080
Date range: 2022-06-09 12:08:53 to 2026-03-01 14:35:06

Columns: ['date', 'is_known', 'is_EU', 'country', 'duration_sec', 'analysis_period', 'trigger_48h', 'trigger_7d', 'trigger_1M', 'date_dt']


## 2. Active Phase Statistics: Post-Audit Analysis

In [59]:
# Calculate baseline metrics (Pre-Audit period)
baseline_mask = df['analysis_period'] == 'Baseline (Pre-Audit)'
baseline = df[baseline_mask]

# Filter for "interesting" calls: EU unknown only (ignore non-EU unknown spam)
baseline_interesting = baseline[(baseline['is_known'] == False) & (baseline['is_EU'] == True)]

# Calculate μ (mean daily intensity)
baseline_start = pd.to_datetime('2022-06-09')
baseline_end = pd.to_datetime('2025-08-02')
baseline_days = (baseline_end - baseline_start).days + 1

μ_base_all = len(baseline) / baseline_days
μ_base_unknown_eu = len(baseline_interesting) / baseline_days

print("=" * 60)
print("BASELINE STATISTICS (Pre-Audit: 2022-06-09 to 2025-08-02)")
print("=" * 60)
print(f"Total days: {baseline_days}")
print(f"Total calls: {len(baseline)}")
print(f"  - Known contacts: {(baseline['is_known'] == True).sum()}")
print(f"  - Unknown EU: {len(baseline_interesting)} ⚠️")
print(f"  - Unknown Non-EU (spam): {((baseline['is_known'] == False) & (baseline['is_EU'] == False)).sum()}")
print(f"\nμ_baseline (all): {μ_base_all:.3f} calls/day")
print(f"μ_baseline (unknown EU): {μ_base_unknown_eu:.3f} calls/day")
print("=" * 60)

BASELINE STATISTICS (Pre-Audit: 2022-06-09 to 2025-08-02)
Total days: 1151
Total calls: 831
  - Known contacts: 555
  - Unknown EU: 154 ⚠️
  - Unknown Non-EU (spam): 83

μ_baseline (all): 0.722 calls/day
μ_baseline (unknown EU): 0.134 calls/day


## 3. Temporal Proximity Analysis: Δt from Trigger Events

In [60]:
# Active phases: Phase 1, 2, 3 combined
active_phases = ['Phase 1: Invisible (RU/BY)',
                 'Phase 2: Visible (DE/UA/Public)',
                 'Phase 3: Manifestation (Performance/AGI)']

results = []
for phase_name in active_phases:
    phase_data = df[df['analysis_period'] == phase_name]
    phase_interesting = phase_data[(phase_data['is_known'] == False) & (phase_data['is_EU'] == True)]

    # Get period dates from config
    start, end = periods[phase_name]
    days = (end - start).days + 1

    μ_phase_all = len(phase_data) / days
    μ_phase_unknown_eu = len(phase_interesting) / days

    # Calculate variance from baseline
    variance_all = ((μ_phase_all - μ_base_all) / μ_base_all * 100) if μ_base_all > 0 else 0
    variance_unknown = ((μ_phase_unknown_eu - μ_base_unknown_eu) / μ_base_unknown_eu * 100) if μ_base_unknown_eu > 0 else 0

    results.append({
        'Phase': phase_name.split(':')[0],
        'Days': days,
        'Total': len(phase_data),
        'Unknown EU': len(phase_interesting),
        'μ_all': f"{μ_phase_all:.2f}",
        'μ_unknown_EU': f"{μ_phase_unknown_eu:.3f}",
        'Δ% (all)': f"{variance_all:+.1f}%",
        'Δ% (unknown EU)': f"{variance_unknown:+.1f}%"
    })

stats_table = pd.DataFrame(results)
print("\nACTIVE PHASE STATISTICS:")
print(stats_table.to_string(index=False))
print(f"\n📊 Reference: μ_baseline (unknown EU) = {μ_base_unknown_eu:.3f} calls/day")


ACTIVE PHASE STATISTICS:
  Phase  Days  Total  Unknown EU μ_all μ_unknown_EU Δ% (all) Δ% (unknown EU)
Phase 1    44     88          18  2.00        0.409  +177.0%         +205.8%
Phase 2    64     79          16  1.23        0.250   +71.0%          +86.9%
Phase 3   102     78          17  0.76        0.167    +5.9%          +24.6%

📊 Reference: μ_baseline (unknown EU) = 0.134 calls/day


## 4. Burst Analysis: Clustering Detection

In [61]:
# Calculate minimum Δt (time delta) to nearest trigger for each call
def min_delta_to_trigger(call_date):
    if pd.isna(call_date):
        return np.nan
    deltas = [(call_date - trigger_dt).total_seconds() / 3600 for trigger_dt in trigger_dates_dt]  # in hours
    return min(deltas, key=abs)

df['Δt_hours'] = df['date_dt'].apply(min_delta_to_trigger)

# Filter for interesting calls (unknown EU) in active phases
active_interesting = df[(df['analysis_period'].isin(active_phases)) &
                        (df['is_known'] == False) &
                        (df['is_EU'] == True)].copy()

# Categorize by proximity
active_interesting['proximity'] = pd.cut(
    active_interesting['Δt_hours'].abs(),
    bins=[0, 48, 168, 360, np.inf],  # 48h, 7d, 15d, rest
    labels=['±48h', '±7d', '±15d', '>15d']
)

print("\nTEMPORAL PROXIMITY ANALYSIS (Unknown EU calls in Active Phases):")
print("=" * 60)
proximity_dist = active_interesting['proximity'].value_counts().sort_index()
print(proximity_dist)
print(f"\nTotal unknown EU calls in active phases: {len(active_interesting)}")
print(f"Calls within ±48h of triggers: {(active_interesting['Δt_hours'].abs() <= 48).sum()}")
print(f"Percentage: {(active_interesting['Δt_hours'].abs() <= 48).sum() / len(active_interesting) * 100:.1f}%")


TEMPORAL PROXIMITY ANALYSIS (Unknown EU calls in Active Phases):
proximity
±48h     7
±7d     14
±15d    10
>15d    20
Name: count, dtype: int64

Total unknown EU calls in active phases: 51
Calls within ±48h of triggers: 7
Percentage: 13.7%


## 5. Entropy of Origin: Geographic Distribution Analysis

In [62]:
# Burst detection: calls within 24h of each other (same country, unknown)
def detect_bursts(data, window_hours=24):
    """Detect burst patterns: multiple calls within window_hours"""
    data = data.sort_values('date_dt').reset_index(drop=True)
    bursts = []

    for country in data['country'].dropna().unique():
        country_calls = data[data['country'] == country].sort_values('date_dt')

        if len(country_calls) < 2:
            continue

        for i in range(len(country_calls) - 1):
            t1 = country_calls.iloc[i]['date_dt']
            t2 = country_calls.iloc[i + 1]['date_dt']
            delta_h = (t2 - t1).total_seconds() / 3600

            if delta_h <= window_hours:
                bursts.append({
                    'Country': country,
                    'First_call': t1,
                    'Second_call': t2,
                    'Δt_hours': delta_h,
                    'Period': country_calls.iloc[i]['analysis_period']
                })

    return pd.DataFrame(bursts)

# Detect bursts in active phases (unknown EU)
bursts_df = detect_bursts(active_interesting)

print("\nBURST ANALYSIS (Unknown EU calls with Δt < 24h):")
print("=" * 60)
if len(bursts_df) > 0:
    print(bursts_df.to_string(index=False))
    print(f"\nTotal burst events detected: {len(bursts_df)}")
else:
    print("No bursts detected (all calls spaced > 24h apart)")

# Check for specific countries with multiple calls
print("\n\nCountry distribution (Unknown EU, Active Phases):")
country_counts = active_interesting['country'].value_counts()
print(country_counts)


BURST ANALYSIS (Unknown EU calls with Δt < 24h):
Country          First_call         Second_call  Δt_hours                                   Period
     DE 2025-08-20 13:20:40 2025-08-20 13:23:04  0.040000               Phase 1: Invisible (RU/BY)
     DE 2025-08-20 13:23:04 2025-08-21 11:26:58 22.065000               Phase 1: Invisible (RU/BY)
     DE 2025-08-21 11:26:58 2025-08-21 11:29:06  0.035556               Phase 1: Invisible (RU/BY)
     DE 2025-08-21 11:29:06 2025-08-21 12:33:58  1.081111               Phase 1: Invisible (RU/BY)
     DE 2025-08-29 11:00:52 2025-08-29 12:45:33  1.744722               Phase 1: Invisible (RU/BY)
     DE 2025-09-10 00:55:21 2025-09-10 17:25:51 16.508333               Phase 1: Invisible (RU/BY)
     DE 2025-09-10 17:25:51 2025-09-11 12:00:24 18.575833               Phase 1: Invisible (RU/BY)
     DE 2025-09-11 12:00:24 2025-09-11 12:01:05  0.011389               Phase 1: Invisible (RU/BY)
     DE 2025-09-11 12:01:05 2025-09-11 12:06:17  0.086667  

In [63]:
# CUSUM (Cumulative Sum) - detects sustained shifts in call intensity
# This is the most powerful method for detecting small but persistent changes

# Create daily call counts
df_daily = df.groupby(df['date_dt'].dt.date).size().reset_index()
df_daily.columns = ['date', 'call_count']
df_daily['date'] = pd.to_datetime(df_daily['date'])
df_daily = df_daily.sort_values('date')

# Calculate CUSUM for unknown EU calls only
df_daily_unknown_eu = df[(df['is_known'] == False) & (df['is_EU'] == True)].groupby(
    df[df['is_known'] == False]['date_dt'].dt.date
).size().reset_index()
df_daily_unknown_eu.columns = ['date', 'call_count']
df_daily_unknown_eu['date'] = pd.to_datetime(df_daily_unknown_eu['date'])

# Fill missing dates with 0
date_range = pd.date_range(start=df['date_dt'].min(), end=df['date_dt'].max(), freq='D')
df_daily_full = pd.DataFrame({'date': date_range})
df_daily_full = df_daily_full.merge(df_daily_unknown_eu, on='date', how='left').fillna(0)

# CUSUM calculation
target = μ_base_unknown_eu  # baseline mean
df_daily_full['deviation'] = df_daily_full['call_count'] - target
df_daily_full['cusum'] = df_daily_full['deviation'].cumsum()

# 3-Sigma Control Chart limits
σ_base = np.sqrt(μ_base_unknown_eu)  # Poisson: variance = mean
UCL = μ_base_unknown_eu + 3 * σ_base  # Upper Control Limit
LCL = max(0, μ_base_unknown_eu - 3 * σ_base)  # Lower Control Limit

# Detect anomalies (days above UCL)
df_daily_full['is_anomaly'] = df_daily_full['call_count'] > UCL

print("\nCUSUM & CONTROL CHART ANALYSIS (Unknown EU calls):")
print("=" * 60)
print(f"Target (μ_baseline): {μ_base_unknown_eu:.4f} calls/day")
print(f"Standard deviation (σ): {σ_base:.4f}")
print(f"Upper Control Limit (UCL = μ + 3σ): {UCL:.4f}")
print(f"Lower Control Limit (LCL = μ - 3σ): {LCL:.4f}")

# Find when CUSUM shows sustained shift
baseline_end = pd.to_datetime('2025-08-02')
df_daily_full['period'] = df_daily_full['date'].apply(
    lambda x: 'Baseline' if x <= baseline_end else 'Active'
)

cusum_baseline_end = df_daily_full[df_daily_full['date'] == baseline_end]['cusum'].values[0] if len(df_daily_full[df_daily_full['date'] == baseline_end]) > 0 else 0
cusum_final = df_daily_full['cusum'].iloc[-1]

print(f"\nCUSUM Analysis:")
print(f"  CUSUM at baseline end (Aug 2, 2025): {cusum_baseline_end:.2f}")
print(f"  CUSUM at final date: {cusum_final:.2f}")
print(f"  Net change: {cusum_final - cusum_baseline_end:+.2f}")

if cusum_final > cusum_baseline_end + 5:
    print(f"  ✓ SUSTAINED INCREASE detected (CUSUM grew by {cusum_final - cusum_baseline_end:.1f})")
else:
    print(f"  ~ No sustained increase detected")

# Count days above UCL
anomaly_days = df_daily_full[df_daily_full['is_anomaly']]
print(f"\n3-Sigma Anomalies (days with calls > {UCL:.2f}):")
print(f"  Total anomaly days: {len(anomaly_days)}")

if len(anomaly_days) > 0:
    print(f"\n  Anomalous dates:")
    for _, row in anomaly_days.iterrows():
        print(f"    {row['date'].date()}: {int(row['call_count'])} calls (expected ≤ {UCL:.2f})")

    # Check if anomalies cluster in active phases
    anomaly_in_active = anomaly_days[anomaly_days['period'] == 'Active']
    print(f"\n  Anomalies in Active phases: {len(anomaly_in_active)} / {len(anomaly_days)}")
    print(f"  Percentage: {len(anomaly_in_active) / len(anomaly_days) * 100:.1f}%")



CUSUM & CONTROL CHART ANALYSIS (Unknown EU calls):
Target (μ_baseline): 0.1338 calls/day
Standard deviation (σ): 0.3658
Upper Control Limit (UCL = μ + 3σ): 1.2311
Lower Control Limit (LCL = μ - 3σ): 0.0000

CUSUM Analysis:
  CUSUM at baseline end (Aug 2, 2025): 0.00
  CUSUM at final date: -182.23
  Net change: -182.23
  ~ No sustained increase detected

3-Sigma Anomalies (days with calls > 1.23):
  Total anomaly days: 0


## 6. Statistical Significance: Correlation Testing

In [64]:
# Geographic Shift Analysis: Country-specific percentage changes
# Detects if specific countries show coordinated increases

print("\nGEOGRAPHIC SHIFT ANALYSIS (Country-level patterns):")
print("=" * 60)

# Calculate country distributions for baseline and active phases
baseline_eu = baseline[(baseline['is_known'] == False) & (baseline['is_EU'] == True)]
active_eu = df[(df['analysis_period'].isin(active_phases)) &
               (df['is_known'] == False) &
               (df['is_EU'] == True)]

baseline_countries = baseline_eu['country'].value_counts(normalize=True) * 100
active_countries = active_eu['country'].value_counts(normalize=True) * 100

active_days_total = sum((periods[p][1] - periods[p][0]).days + 1 for p in active_phases)

# Calculate shifts
all_countries = set(baseline_countries.index) | set(active_countries.index)
shifts = []

for country in all_countries:
    baseline_pct = baseline_countries.get(country, 0)
    active_pct = active_countries.get(country, 0)

    baseline_count = baseline_eu[baseline_eu['country'] == country].shape[0]
    active_count = active_eu[active_eu['country'] == country].shape[0]

    # Calculate rate per day
    baseline_rate = baseline_count / baseline_days
    active_rate = active_count / active_days_total

    # Robust rate-change text for zero-baseline countries
    if baseline_rate == 0 and active_rate > 0:
        rate_change_str = "new"
    elif baseline_rate == 0 and active_rate == 0:
        rate_change_str = "0%"
    else:
        rate_change_str = f"{((active_rate / baseline_rate - 1) * 100):+.0f}%"

    shifts.append({
        'Country': country,
        'Baseline%': f"{baseline_pct:.1f}",
        'Active%': f"{active_pct:.1f}",
        'Δ%': f"{active_pct - baseline_pct:+.1f}",
        'Base_rate': f"{baseline_rate:.3f}",
        'Active_rate': f"{active_rate:.3f}",
        'Rate_change': rate_change_str
    })

shifts_df = pd.DataFrame(shifts)
shifts_df['Δ%_numeric'] = shifts_df['Δ%'].str.replace('+', '').astype(float)
shifts_df = shifts_df.sort_values('Δ%_numeric', ascending=False)

print("\nCountry-level changes (Unknown EU calls):\n")
print(shifts_df[['Country', 'Baseline%', 'Active%', 'Δ%', 'Base_rate', 'Active_rate', 'Rate_change']].to_string(index=False))

# Highlight significant shifts
print(f"\n\nKey observations:")
significant_increases = shifts_df[shifts_df['Δ%_numeric'] > 5]
if len(significant_increases) > 0:
    print(f"\nCountries with major increases (Δ > +5%):")
    for _, row in significant_increases.iterrows():
        print(f"  • {row['Country']}: {row['Baseline%']}% → {row['Active%']}% ({row['Rate_change']} rate change)")

significant_decreases = shifts_df[shifts_df['Δ%_numeric'] < -5]
if len(significant_decreases) > 0:
    print(f"\nCountries with major decreases (Δ < -5%):")
    for _, row in significant_decreases.iterrows():
        print(f"  • {row['Country']}: {row['Baseline%']}% → {row['Active%']}% ({row['Rate_change']} rate change)")

# Check for VOIP gateway patterns (Poland, Italy, Austria mentioned in text)
voip_countries = ['PL', 'IT', 'AT']  # Poland, Italy, Austria
voip_shifts = shifts_df[shifts_df['Country'].isin(voip_countries)]

if len(voip_shifts) > 0:
    print(f"\n\nSuspected VOIP Gateway countries (PL/IT/AT):")
    print(voip_shifts[['Country', 'Baseline%', 'Active%', 'Δ%', 'Rate_change']].to_string(index=False))

    total_voip_active = sum([active_countries.get(c, 0) for c in voip_countries])
    total_voip_baseline = sum([baseline_countries.get(c, 0) for c in voip_countries])

    print(f"\n  Combined VOIP countries:")
    print(f"    Baseline: {total_voip_baseline:.1f}%")
    print(f"    Active: {total_voip_active:.1f}%")
    print(f"    Change: {total_voip_active - total_voip_baseline:+.1f}%")



GEOGRAPHIC SHIFT ANALYSIS (Country-level patterns):

Country-level changes (Unknown EU calls):

Country Baseline% Active%   Δ% Base_rate Active_rate Rate_change
     PL       0.0     5.9 +5.9     0.000       0.014         new
     AT       0.0     3.9 +3.9     0.000       0.010         new
     DE      88.3    90.2 +1.9     0.118       0.219        +85%
     NL       0.6     0.0 -0.6     0.001       0.000       -100%
     RO       0.6     0.0 -0.6     0.001       0.000       -100%
     PT       1.9     0.0 -1.9     0.003       0.000       -100%
     EE       3.2     0.0 -3.2     0.004       0.000       -100%
     DK       5.2     0.0 -5.2     0.007       0.000       -100%


Key observations:

Countries with major increases (Δ > +5%):
  • PL: 0.0% → 5.9% (new rate change)

Countries with major decreases (Δ < -5%):
  • DK: 5.2% → 0.0% (-100% rate change)


Suspected VOIP Gateway countries (PL/IT/AT):
Country Baseline% Active%   Δ% Rate_change
     PL       0.0     5.9 +5.9         new
 

In [65]:
# Lag Time Analysis: Time from trigger event to first anomalous call
# This measures reaction time - critical for proving causality

print("\nLAG TIME ANALYSIS (Trigger → First Unknown EU Call):")
print("=" * 60)

lag_times = []

for i, trigger_dt in enumerate(trigger_dates_dt):
    # Find first unknown EU call AFTER this trigger
    calls_after = df[(df['date_dt'] >= trigger_dt) &
                     (df['is_known'] == False) &
                     (df['is_EU'] == True)].sort_values('date_dt')

    if len(calls_after) > 0:
        first_call_dt = calls_after.iloc[0]['date_dt']
        lag_hours = (first_call_dt - trigger_dt).total_seconds() / 3600

        # Count calls in 48h window
        window_calls = df[(df['date_dt'] >= trigger_dt) &
                         (df['date_dt'] <= trigger_dt + pd.Timedelta(hours=48)) &
                         (df['is_known'] == False) &
                         (df['is_EU'] == True)]

        lag_times.append({
            'Trigger': trigger_dt.date(),
            'First_call': first_call_dt,
            'Lag_hours': lag_hours,
            'Calls_48h': len(window_calls)
        })

        print(f"\nTrigger: {trigger_dt.date()}")
        print(f"  First call: {first_call_dt}")
        print(f"  Lag time: {lag_hours:.1f} hours ({lag_hours/24:.1f} days)")
        print(f"  Calls in next 48h: {len(window_calls)}")

        if lag_hours < 48:
            print(f"  ✓ IMMEDIATE REACTION (< 48h)")
        elif lag_hours < 168:
            print(f"  → Weekly window (< 7d)")

lag_df = pd.DataFrame(lag_times)

if len(lag_df) > 0:
    print(f"\n\nSummary Statistics:")
    print(f"  Mean lag time: {lag_df['Lag_hours'].mean():.1f} hours")
    print(f"  Median lag time: {lag_df['Lag_hours'].median():.1f} hours")
    print(f"  Triggers with <48h reaction: {(lag_df['Lag_hours'] < 48).sum()} / {len(lag_df)}")
    print(f"  Average calls per trigger (48h window): {lag_df['Calls_48h'].mean():.2f}")

    # Statistical test: Is lag time distribution consistent with random?
    # If calls were random, lag should be uniformly distributed
    # If calls are reactive, lag should cluster near 0

    short_lag_count = (lag_df['Lag_hours'] < 48).sum()
    total_triggers = len(lag_df)

    # Probability of having this many short lags by chance
    # Assuming calls are spread evenly over 30 days, P(call in 48h) ≈ 48/(30*24) = 0.067
    from scipy.stats import binom
    p_random = 48 / (30 * 24)  # probability of random call in 48h window
    p_clustering = 1 - binom.cdf(short_lag_count - 1, total_triggers, p_random)

    print(f"\n  Clustering test:")
    print(f"    P(≥{short_lag_count} triggers with <48h lag by chance) = {p_clustering:.4f}")

    if p_clustering < 0.05:
        print(f"    ✓ SIGNIFICANT clustering near triggers (p < 0.05)")



LAG TIME ANALYSIS (Trigger → First Unknown EU Call):

Trigger: 2025-08-03
  First call: 2025-08-04 11:27:32
  Lag time: 35.5 hours (1.5 days)
  Calls in next 48h: 2
  ✓ IMMEDIATE REACTION (< 48h)

Trigger: 2025-09-16
  First call: 2025-09-21 20:55:16
  Lag time: 140.9 hours (5.9 days)
  Calls in next 48h: 0
  → Weekly window (< 7d)

Trigger: 2025-09-19
  First call: 2025-09-21 20:55:16
  Lag time: 68.9 hours (2.9 days)
  Calls in next 48h: 0
  → Weekly window (< 7d)

Trigger: 2025-11-19
  First call: 2025-11-24 13:55:34
  Lag time: 133.9 hours (5.6 days)
  Calls in next 48h: 0
  → Weekly window (< 7d)

Trigger: 2026-01-07
  First call: 2026-01-16 09:43:47
  Lag time: 225.7 hours (9.4 days)
  Calls in next 48h: 0

Trigger: 2026-01-15
  First call: 2026-01-16 09:43:47
  Lag time: 33.7 hours (1.4 days)
  Calls in next 48h: 1
  ✓ IMMEDIATE REACTION (< 48h)

Trigger: 2026-02-24
  First call: 2026-02-27 16:25:34
  Lag time: 88.4 hours (3.7 days)
  Calls in next 48h: 0
  → Weekly window (< 7

In [66]:
# Exact Poisson Test for specific spike events
# Calculate probability that observed spikes occurred by chance

from scipy.stats import poisson

print("\nEXACT POISSON PROBABILITY FOR SPIKE EVENTS:")
print("=" * 60)

# Analyze specific high-activity days
spike_dates = df_daily_full[df_daily_full['call_count'] >= 2].sort_values('call_count', ascending=False)

if len(spike_dates) > 0:
    print(f"Days with ≥2 unknown EU calls (baseline μ = {μ_base_unknown_eu:.4f}):\n")

    for _, row in spike_dates.head(10).iterrows():
        k = int(row['call_count'])  # observed calls
        λ = μ_base_unknown_eu  # expected rate

        # P(X ≥ k) = 1 - P(X < k) = 1 - CDF(k-1)
        p_value = 1 - poisson.cdf(k - 1, λ)

        # Format p-value with scientific notation if very small
        if p_value < 0.0001:
            p_str = f"{p_value:.2e}"
        else:
            p_str = f"{p_value:.6f}"

        # Check if in trigger window
        trigger_status = ""
        for trigger_dt in trigger_dates_dt:
            if abs((row['date'] - trigger_dt).days) <= 2:
                trigger_status = f"  [±48h from {trigger_dt.date()}]"
                break

        significance = ""
        if p_value < 0.001:
            significance = "  ✓ EXTREMELY SIGNIFICANT"
        elif p_value < 0.01:
            significance = "  ✓ VERY SIGNIFICANT"
        elif p_value < 0.05:
            significance = "  ✓ SIGNIFICANT"

        print(f"  {row['date'].date()}: {k} calls")
        print(f"    P(X ≥ {k} | λ={λ:.4f}) = {p_str}{significance}{trigger_status}")
        print()

    # Calculate combined probability for all active phase
    active_phase_data = df_daily_full[df_daily_full['period'] == 'Active']
    total_active_calls = active_phase_data['call_count'].sum()
    active_days = len(active_phase_data)
    expected_active_calls = μ_base_unknown_eu * active_days

    p_value_total = 1 - poisson.cdf(int(total_active_calls) - 1, expected_active_calls)

    print(f"\nCombined Active Phase Analysis:")
    print(f"  Active phase days: {active_days}")
    print(f"  Expected calls (under baseline): {expected_active_calls:.2f}")
    print(f"  Observed calls: {int(total_active_calls)}")
    print(f"  P(X ≥ {int(total_active_calls)}) = {p_value_total:.2e}")

    if p_value_total < 1e-10:
        print(f"  ✓ Probability of randomness: < 1 in 10 billion")
else:
    print("  No days with multiple unknown EU calls found")



EXACT POISSON PROBABILITY FOR SPIKE EVENTS:
  No days with multiple unknown EU calls found


## 5.5. Advanced Statistical Tests: CUSUM & Control Charts

In [67]:
# Calculate Shannon entropy for geographic distribution
from scipy.stats import entropy

def calc_entropy(series):
    """Calculate Shannon entropy of categorical distribution"""
    counts = series.value_counts(normalize=True)
    return entropy(counts, base=2)

# Compare baseline vs active geographic entropy (all calls)
baseline_countries = baseline['country'].dropna()
active_all = df[df['analysis_period'].isin(active_phases)]
active_countries = active_all['country'].dropna()

H_baseline = calc_entropy(baseline_countries)
H_active = calc_entropy(active_countries)

print("\nGEOGRAPHIC ENTROPY ANALYSIS:")
print("=" * 60)
print(f"Shannon Entropy (H):")
print(f"  Baseline: {H_baseline:.3f} bits")
print(f"  Active:   {H_active:.3f} bits")
print(f"  ΔH:       {H_active - H_baseline:+.3f} bits")

# Show country distribution comparison
print("\n\nTop countries (Baseline, all calls):")
print(baseline_countries.value_counts().head(10))

print("\n\nTop countries (Active phases, all calls):")
print(active_countries.value_counts().head(10))

# Calculate percentage shifts
baseline_dist = baseline_countries.value_counts(normalize=True)
active_dist = active_countries.value_counts(normalize=True)

print("\n\nKey shifts (Active vs Baseline):")
all_countries = set(baseline_dist.index) | set(active_dist.index)
shifts = []
for country in all_countries:
    b = baseline_dist.get(country, 0) * 100
    a = active_dist.get(country, 0) * 100
    if abs(a - b) > 1:  # Only show significant shifts
        shifts.append({
            'Country': country,
            'Baseline%': f"{b:.1f}",
            'Active%': f"{a:.1f}",
            'Δ%': f"{a - b:+.1f}"
        })

shifts_df = pd.DataFrame(shifts).sort_values('Δ%', ascending=False, key=lambda x: x.str.replace('+', '').astype(float))
print(shifts_df.to_string(index=False))


GEOGRAPHIC ENTROPY ANALYSIS:
Shannon Entropy (H):
  Baseline: 0.974 bits
  Active:   0.406 bits
  ΔH:       -0.568 bits


Top countries (Baseline, all calls):
country
DE        590
Non-EU    175
DK         10
EE          5
PT          3
RO          1
NL          1
Name: count, dtype: int64


Top countries (Active phases, all calls):
country
DE        218
Non-EU      9
PL          3
AT          2
Name: count, dtype: int64


Key shifts (Active vs Baseline):
Country Baseline% Active%    Δ%
     DE      75.2    94.0 +18.8
     PL       0.0     1.3  +1.3
     DK       1.3     0.0  -1.3
 Non-EU      22.3     3.9 -18.4


## 7. Signal-to-Silence Ratio: Post-Publication Dynamics

In [68]:
# Statistical test: Chi-square for unknown EU call distribution
from scipy.stats import chi2_contingency, poisson

# Get active phase data
active_all = df[df['analysis_period'].isin(active_phases)]
active_interesting = active_all[(active_all['is_known'] == False) & (active_all['is_EU'] == True)]

# Create contingency table: Baseline vs Active, Unknown EU vs Others
contingency = pd.DataFrame({
    'Unknown EU': [
        len(baseline_interesting),
        len(active_interesting)
    ],
    'Other calls': [
        len(baseline) - len(baseline_interesting),
        len(active_all) - len(active_interesting)
    ]
}, index=['Baseline', 'Active'])

print("\nSTATISTICAL SIGNIFICANCE TESTING:")
print("=" * 60)
print("Contingency Table:")
print(contingency)

chi2, p_value, dof, expected = chi2_contingency(contingency)
print(f"\nChi-square test:")
print(f"  χ² = {chi2:.4f}")
print(f"  p-value = {p_value:.6f}")
print(f"  DOF = {dof}")

if p_value < 0.001:
    print(f"\n  ✓ HIGHLY SIGNIFICANT (p < 0.001)")
    print(f"    The distribution of unknown EU calls changed significantly")
elif p_value < 0.05:
    print(f"\n  ✓ SIGNIFICANT (p < 0.05)")
else:
    print(f"\n  ✗ Not statistically significant")

# Poisson rate test for baseline vs Phase 1
phase1 = df[df['analysis_period'] == 'Phase 1: Invisible (RU/BY)']
phase1_interesting = phase1[(phase1['is_known'] == False) & (phase1['is_EU'] == True)]
phase1_days = (periods['Phase 1: Invisible (RU/BY)'][1] - periods['Phase 1: Invisible (RU/BY)'][0]).days + 1

λ_baseline = len(baseline_interesting) / baseline_days  # events per day
λ_phase1 = len(phase1_interesting) / phase1_days

print(f"\n\nPoisson Rate Comparison (Unknown EU calls/day):")
print(f"  λ_baseline = {λ_baseline:.4f}")
print(f"  λ_phase1   = {λ_phase1:.4f}")
print(f"  Rate increase: {(λ_phase1 / λ_baseline - 1) * 100:.1f}%")

# Expected number in Phase 1 under baseline rate
expected_phase1 = λ_baseline * phase1_days
observed_phase1 = len(phase1_interesting)
print(f"\n  Expected (under baseline): {expected_phase1:.2f} calls")
print(f"  Observed: {observed_phase1} calls")
print(f"  Excess: {observed_phase1 - expected_phase1:+.2f} calls")


STATISTICAL SIGNIFICANCE TESTING:
Contingency Table:
          Unknown EU  Other calls
Baseline         154          677
Active            51          194

Chi-square test:
  χ² = 0.5007
  p-value = 0.479186
  DOF = 1

  ✗ Not statistically significant


Poisson Rate Comparison (Unknown EU calls/day):
  λ_baseline = 0.1338
  λ_phase1   = 0.4091
  Rate increase: 205.8%

  Expected (under baseline): 5.89 calls
  Observed: 18 calls
  Excess: +12.11 calls


## 8. Anomaly Scoring: Individual Call Assessment

In [69]:
# Analyze call patterns before/after key events
# Focus on major trigger: Feb 27, 2026 (Final README + Austria anomaly)

trigger_feb27 = pd.to_datetime('2026-02-27')
window_before = df[(df['date_dt'] >= trigger_feb27 - pd.Timedelta(days=7)) &
                   (df['date_dt'] < trigger_feb27)]
window_after = df[(df['date_dt'] >= trigger_feb27) &
                  (df['date_dt'] <= trigger_feb27 + pd.Timedelta(days=7))]

before_interesting = window_before[(window_before['is_known'] == False) & (window_before['is_EU'] == True)]
after_interesting = window_after[(window_after['is_known'] == False) & (window_after['is_EU'] == True)]

print("\nSIGNAL-TO-SILENCE ANALYSIS (Feb 27 README event):")
print("=" * 60)
print(f"7 days BEFORE Feb 27:")
print(f"  Total calls: {len(window_before)}")
print(f"  Unknown EU: {len(before_interesting)}")

print(f"\n7 days AFTER Feb 27:")
print(f"  Total calls: {len(window_after)}")
print(f"  Unknown EU: {len(after_interesting)}")

print(f"\nSignal-to-Silence ratio (unknown EU):")
ratio = len(before_interesting) / len(after_interesting) if len(after_interesting) > 0 else np.inf
print(f"  Before/After = {ratio:.2f}")

if ratio > 2:
    print(f"  ✓ SILENCE after publication detected (ratio > 2)")
elif ratio < 0.5:
    print(f"  ⚠️ INCREASE after publication (ratio < 0.5)")
else:
    print(f"  ~ No significant change")

# Show actual calls around Feb 27
print("\n\nCalls within ±2 days of Feb 27, 2026:")
window_tight = df[(df['date_dt'] >= trigger_feb27 - pd.Timedelta(days=2)) &
                  (df['date_dt'] <= trigger_feb27 + pd.Timedelta(days=2))]
window_tight_int = window_tight[(window_tight['is_known'] == False) & (window_tight['is_EU'] == True)]
if len(window_tight_int) > 0:
    print(window_tight_int[['date_dt', 'country', 'duration_sec']].to_string(index=False))
else:
    print("  (No unknown EU calls in this window)")


SIGNAL-TO-SILENCE ANALYSIS (Feb 27 README event):
7 days BEFORE Feb 27:
  Total calls: 3
  Unknown EU: 1

7 days AFTER Feb 27:
  Total calls: 10
  Unknown EU: 2

Signal-to-Silence ratio (unknown EU):
  Before/After = 0.50
  ~ No significant change


Calls within ±2 days of Feb 27, 2026:
            date_dt country  duration_sec
2026-02-27 16:25:34      AT             0
2026-02-27 16:26:17      AT             0


## 9. Summary Report: Analysis #1 Findings

In [70]:
# Calculate anomaly score for each call based on multiple factors
def calculate_anomaly_score(row):
    """
    Anomaly score components:
    1. Unknown contact: +2 points
    2. EU origin: +1 point
    3. In active phase: +2 points
    4. Within 48h of trigger: +3 points
    5. From rare country (< 5 calls total): +2 points
    6. Very short duration (< 10 sec): +1 point
    """
    score = 0

    if row['is_known'] == False:
        score += 2

    if row['is_EU'] == True:
        score += 1

    if row['analysis_period'] in active_phases:
        score += 2

    if row['trigger_48h'] == True:
        score += 3

    # Check if rare country (EU only)
    if pd.notna(row['country']) and row['country'] != 'Non-EU':
        country_count = df[df['country'] == row['country']].shape[0]
        if country_count < 5:
            score += 2

    if pd.notna(row['duration_sec']) and row['duration_sec'] < 10:
        score += 1

    return score

df['anomaly_score'] = df.apply(calculate_anomaly_score, axis=1)

# Show top anomalies (focus on unknown EU calls)
print("\nTOP ANOMALOUS CALLS (unknown EU, score ≥ 6):")
print("=" * 60)
high_anomaly = df[(df['is_known'] == False) &
                  (df['is_EU'] == True) &
                  (df['anomaly_score'] >= 6)].sort_values('anomaly_score', ascending=False)

if len(high_anomaly) > 0:
    display_cols = ['date_dt', 'country', 'analysis_period', 'trigger_48h', 'duration_sec', 'anomaly_score']
    print(high_anomaly[display_cols].head(20).to_string(index=False))
    print(f"\nTotal high-anomaly unknown EU calls: {len(high_anomaly)}")
else:
    print("No unknown EU calls with score ≥ 6")

# Distribution of scores for unknown EU calls
unknown_eu = df[(df['is_known'] == False) & (df['is_EU'] == True)]
print("\n\nAnomaly Score Distribution (Unknown EU calls only):")
print(unknown_eu['anomaly_score'].value_counts().sort_index(ascending=False))


TOP ANOMALOUS CALLS (unknown EU, score ≥ 6):
            date_dt country                          analysis_period  trigger_48h  duration_sec  anomaly_score
2026-02-27 16:26:17      AT Phase 3: Manifestation (Performance/AGI)         True             0             11
2026-02-27 16:25:34      AT Phase 3: Manifestation (Performance/AGI)         True             0             11
2025-08-04 14:02:21      PL               Phase 1: Invisible (RU/BY)         True             0             11
2025-09-21 20:55:16      DE          Phase 2: Visible (DE/UA/Public)         True             0              9
2025-08-04 11:27:32      DE               Phase 1: Invisible (RU/BY)         True             0              9
2026-02-23 09:23:37      DE Phase 3: Manifestation (Performance/AGI)         True            44              8
2026-01-16 09:43:47      DE Phase 3: Manifestation (Performance/AGI)         True           462              8
2026-01-05 09:16:54      DE Phase 3: Manifestation (Performance/AG

In [71]:
# Generate comprehensive summary report
print("\n" + "=" * 80)
print("ANALYSIS #1: STATISTICAL CORRELATION BETWEEN AUDIT AND CALL ANOMALIES")
print("=" * 80)

print(f"\n📅 OBSERVATION PERIOD: {df['date_dt'].min().date()} to {df['date_dt'].max().date()}")
print(f"   Total days: {(df['date_dt'].max() - df['date_dt'].min()).days}")
print(f"   Total calls: {len(df)}")

print(f"\n📊 BASELINE METRICS (Pre-Audit: 2022-06-09 to 2025-08-02):")
print(f"   μ_baseline = {μ_base_all:.3f} calls/day (all)")
print(f"   μ_baseline = {μ_base_unknown_eu:.3f} calls/day (unknown EU) ⚠️")

print(f"\n🚀 PHASE 1 SPIKE (Invisible/RU-BY: Aug 3-Sep 15, 2025):")
print(f"   μ_phase1 = {λ_phase1:.3f} calls/day (unknown EU)")
print(f"   Increase: {(λ_phase1 / μ_base_unknown_eu - 1) * 100:+.1f}%")

print(f"\n🎯 TRIGGER PROXIMITY (Unknown EU calls in active phases):")
print(f"   Total: {len(active_interesting)}")
print(f"   Within ±48h of triggers: {(active_interesting['Δt_hours'].abs() <= 48).sum()}")
print(f"   Percentage: {(active_interesting['Δt_hours'].abs() <= 48).sum() / len(active_interesting) * 100:.1f}%")

print(f"\n📡 GEOGRAPHIC ENTROPY:")
print(f"   Baseline: H = {H_baseline:.3f} bits")
print(f"   Active:   H = {H_active:.3f} bits")
print(f"   Change:   ΔH = {H_active - H_baseline:+.3f} bits")

print(f"\n📈 STATISTICAL SIGNIFICANCE:")
print(f"   Chi-square p-value: {p_value:.6f}")

if p_value < 0.001:
    significance_text = "HIGHLY SIGNIFICANT (p < 0.001)"
elif p_value < 0.05:
    significance_text = "SIGNIFICANT (p < 0.05)"
else:
    significance_text = "NOT SIGNIFICANT (p ≥ 0.05)"

print(f"   Verdict: {significance_text}")

print(f"\n🔇 SIGNAL-TO-SILENCE (Feb 27 event):")
print(f"   7d before: {len(before_interesting)} unknown EU calls")
print(f"   7d after:  {len(after_interesting)} unknown EU calls")
print(f"   Ratio:     {ratio:.2f}")

print(f"\n⚠️ HIGH-ANOMALY CALLS (score ≥ 7): {len(high_anomaly)}")

print("\n" + "=" * 80)
print("CONCLUSION:")
print("=" * 80)
print("Primary inferential test in this report:")
print(f"  p = {p_value:.6f}")

if p_value < 0.001:
    print("Result: Very strong statistical evidence of a distribution shift.")
elif p_value < 0.05:
    print("Result: Statistical evidence of a distribution shift.")
else:
    print("Result: Insufficient evidence to claim a statistically significant distribution shift.")

print("\nInterpretation note:")
print("  Operational indicators (phase spike, lag clustering, geography shifts) may still be relevant")
print("  but should be treated as exploratory if inferential p-value is not significant.")
print("=" * 80)


ANALYSIS #1: STATISTICAL CORRELATION BETWEEN AUDIT AND CALL ANOMALIES

📅 OBSERVATION PERIOD: 2022-06-09 to 2026-03-01
   Total days: 1361
   Total calls: 1080

📊 BASELINE METRICS (Pre-Audit: 2022-06-09 to 2025-08-02):
   μ_baseline = 0.722 calls/day (all)
   μ_baseline = 0.134 calls/day (unknown EU) ⚠️

🚀 PHASE 1 SPIKE (Invisible/RU-BY: Aug 3-Sep 15, 2025):
   μ_phase1 = 0.409 calls/day (unknown EU)
   Increase: +205.8%

🎯 TRIGGER PROXIMITY (Unknown EU calls in active phases):
   Total: 51
   Within ±48h of triggers: 7
   Percentage: 13.7%

📡 GEOGRAPHIC ENTROPY:
   Baseline: H = 0.974 bits
   Active:   H = 0.406 bits
   Change:   ΔH = -0.568 bits

📈 STATISTICAL SIGNIFICANCE:
   Chi-square p-value: 0.479186
   Verdict: NOT SIGNIFICANT (p ≥ 0.05)

🔇 SIGNAL-TO-SILENCE (Feb 27 event):
   7d before: 1 unknown EU calls
   7d after:  2 unknown EU calls
   Ratio:     0.50

⚠️ HIGH-ANOMALY CALLS (score ≥ 7): 42

CONCLUSION:
Primary inferential test in this report:
  p = 0.479186
Result: Insuff

## 10. Final Executive Summary (Audit-ready)

This section condenses results into **inferential evidence** (formal hypothesis testing) and **exploratory signals** (operational patterns).

In [72]:
# Compact audit-ready summary

within_48h = int((active_interesting['Δt_hours'].abs() <= 48).sum())
total_active_unknown_eu = int(len(active_interesting))
trigger_share_48h = (within_48h / total_active_unknown_eu * 100) if total_active_unknown_eu > 0 else 0.0

tier_inferential = "confirmed" if p_value < 0.05 else "not-confirmed"
tier_lag = "positive" if 'p_clustering' in globals() and p_clustering < 0.05 else "weak"
tier_geo = "positive" if (total_voip_active - total_voip_baseline) >= 5 else "weak"

audit_summary = pd.DataFrame([
    {
        'Layer': 'Inferential test (chi-square)',
        'Metric': f'p = {p_value:.6f}',
        'Interpretation': 'Statistically significant shift' if p_value < 0.05 else 'No statistically significant shift'
    },
    {
        'Layer': 'Lag clustering near triggers',
        'Metric': f'p = {p_clustering:.4f}' if 'p_clustering' in globals() else 'n/a',
        'Interpretation': 'Clustering near trigger times' if tier_lag == 'positive' else 'No strong clustering evidence'
    },
    {
        'Layer': 'Trigger proximity (±48h)',
        'Metric': f'{within_48h}/{total_active_unknown_eu} ({trigger_share_48h:.1f}%)',
        'Interpretation': 'Substantial immediate overlap with trigger windows' if trigger_share_48h >= 15 else 'Limited immediate overlap'
    },
    {
        'Layer': 'Geographic concentration shift',
        'Metric': f'ΔH = {H_active - H_baseline:+.3f}, VOIP Δ = {total_voip_active - total_voip_baseline:+.1f} pp',
        'Interpretation': 'Concentration and country-mix shift observed' if tier_geo == 'positive' else 'Small geographic shift'
    }
])

print("\n" + "=" * 80)
print("FINAL EXECUTIVE SUMMARY")
print("=" * 80)
print(audit_summary.to_string(index=False))

print("\nOverall evidence tiering:")
print(f"  • Inferential evidence: {tier_inferential}")
print(f"  • Operational/behavioral signals: {'positive' if (tier_lag == 'positive' or tier_geo == 'positive' or trigger_share_48h >= 15) else 'mixed'}")

if p_value < 0.05:
    print("\nBottom line: Formal statistical evidence supports a distribution shift.")
else:
    print("\nBottom line: Formal statistical evidence is insufficient; treat findings as exploratory but structured signals.")
print("=" * 80)


FINAL EXECUTIVE SUMMARY
                         Layer                        Metric                               Interpretation
 Inferential test (chi-square)                  p = 0.479186           No statistically significant shift
  Lag clustering near triggers                    p = 0.0129                Clustering near trigger times
      Trigger proximity (±48h)                  7/51 (13.7%)                    Limited immediate overlap
Geographic concentration shift ΔH = -0.568, VOIP Δ = +9.8 pp Concentration and country-mix shift observed

Overall evidence tiering:
  • Inferential evidence: not-confirmed
  • Operational/behavioral signals: positive

Bottom line: Formal statistical evidence is insufficient; treat findings as exploratory but structured signals.


In [73]:
# Export analysis results
output_file = 'calls-analysis-complete.csv'
df.to_csv(output_file, index=False)
print(f"\n✓ Complete analysis saved to: {output_file}")
print(f"\nDataset summary:")
print(f"  Total rows: {len(df)}")
print(f"  Unknown EU calls: {len(df[(df['is_known'] == False) & (df['is_EU'] == True)])}")
print(f"  Unknown Non-EU calls (spam): {len(df[(df['is_known'] == False) & (df['is_EU'] == False)])}")
print(f"  High-anomaly calls (score ≥ 7): {len(df[df['anomaly_score'] >= 7])}")


✓ Complete analysis saved to: calls-analysis-complete.csv

Dataset summary:
  Total rows: 1080
  Unknown EU calls: 205
  Unknown Non-EU calls (spam): 84
  High-anomaly calls (score ≥ 7): 26
